# Homework A Coding Question

This notebook contains the source code for Question 4. It builds a visibility graph for a point robot among polygonal obstacles and solves the path with A*.

In [ ]:
import math
import heapq
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


OBSTACLES = [
    [(2.0, 5.7), (2.6, 7.45), (4.25, 7.4), (4.65, 5.55), (3.2, 6.65)],
    [(5.75, 5.45), (6.35, 7.05), (7.25, 5.2)],
    [(3.2, 3.2), (4.75, 3.55), (4.15, 2.75), (4.55, 1.75)],
]

START = (3.35, 1.25)
GOAL = (7.45, 7.35)


def as_array(point):
    return np.array(point, dtype=float)


def segment_intersection(a, b, c, d):
    def orient(p, q, r):
        return np.cross(q - p, r - p)

    a, b, c, d = map(as_array, (a, b, c, d))
    o1 = orient(a, b, c)
    o2 = orient(a, b, d)
    o3 = orient(c, d, a)
    o4 = orient(c, d, b)

    if np.isclose(o1, 0.0) and np.isclose(o2, 0.0) and np.isclose(o3, 0.0) and np.isclose(o4, 0.0):
        return False
    return (o1 * o2 < 0.0) and (o3 * o4 < 0.0)


def point_in_polygon(point, polygon):
    x, y = point
    inside = False
    for i in range(len(polygon)):
        x1, y1 = polygon[i]
        x2, y2 = polygon[(i + 1) % len(polygon)]
        cond = ((y1 > y) != (y2 > y))
        if cond:
            x_hit = (x2 - x1) * (y - y1) / (y2 - y1) + x1
            if x < x_hit:
                inside = not inside
    return inside


def visible(p, q, obstacles):
    midpoint = 0.5 * (as_array(p) + as_array(q))
    for poly in obstacles:
        for i in range(len(poly)):
            a = poly[i]
            b = poly[(i + 1) % len(poly)]
            if p in (a, b) or q in (a, b):
                continue
            if segment_intersection(p, q, a, b):
                return False
        if point_in_polygon(midpoint, poly):
            return False
    return True


def euclidean(p, q):
    return float(np.linalg.norm(as_array(p) - as_array(q)))


def build_visibility_graph(obstacles, start, goal):
    nodes = [start, goal]
    for poly in obstacles:
        nodes.extend(poly)
    graph = {node: [] for node in nodes}
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            if visible(nodes[i], nodes[j], obstacles):
                cost = euclidean(nodes[i], nodes[j])
                graph[nodes[i]].append((nodes[j], cost))
                graph[nodes[j]].append((nodes[i], cost))
    return graph


def astar(graph, start, goal):
    queue = [(euclidean(start, goal), 0.0, start)]
    parent = {start: None}
    cost_so_far = {start: 0.0}

    while queue:
        _, current_cost, current = heapq.heappop(queue)
        if current == goal:
            break
        if current_cost > cost_so_far[current]:
            continue
        for nxt, edge_cost in graph[current]:
            new_cost = current_cost + edge_cost
            if nxt not in cost_so_far or new_cost < cost_so_far[nxt]:
                cost_so_far[nxt] = new_cost
                priority = new_cost + euclidean(nxt, goal)
                heapq.heappush(queue, (priority, new_cost, nxt))
                parent[nxt] = current

    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse()
    return path, cost_so_far[goal]


graph = build_visibility_graph(OBSTACLES, START, GOAL)
path, length = astar(graph, START, GOAL)
print("Path:")
for node in path:
    print(node)
print("Length:", round(length, 6))


fig, ax = plt.subplots(figsize=(6, 6))
for poly in OBSTACLES:
    patch = plt.Polygon(poly, closed=True, facecolor="#dbe7ef", edgecolor="black")
    ax.add_patch(patch)

path_arr = np.array(path)
ax.plot(path_arr[:, 0], path_arr[:, 1], "-o", color="#cc4c4c", lw=2)
ax.scatter(*START, s=50, color="#1b9e77")
ax.scatter(*GOAL, s=50, color="#377eb8")
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Homework A4 path")
ax.grid(alpha=0.2)
plt.show()
